# Exercise 3: The Same Agent with LangChain

In Exercise 2 you wrote the ReAct loop by hand: call the model, check for tool calls, run the
tool, append the result, repeat. That loop is the same for almost every agent, so frameworks
ship it for you.

Here you build the **same agent**, with the **same three tools**, against the **same model**, using
**LangChain**. The whole `run_agent` function from Exercise 2 collapses into one call to
`create_agent`. The point of this exercise is to see exactly what the framework does for you, and
what you give up in exchange.

## What changes

| By hand (Exercise 2) | With LangChain (here) |
|---|---|
| You write the JSON schema for each tool | The `@tool` decorator generates it from the function |
| You write the `for`-loop, the `tool_calls` check, the dispatch | `create_agent` runs the loop |
| You append messages to the history yourself | The framework manages the message list |
| Raw `dict` messages | Typed message objects (`HumanMessage`, `AIMessage`, `ToolMessage`) |

## Setup check

Verify the LangChain + Ollama packages import and the server is reachable. If this fails, run
`uv sync` again and make sure Ollama is running (`ollama list`).

In [ ]:
import math
import os
import numexpr
from ddgs import DDGS

from langchain_core.tools import tool
from langchain_ollama import ChatOllama
from langchain.agents import create_agent

MODEL = "qwen3.5:4b"
print("imports OK, model:", MODEL)

## Step 1: Define the tools with `@tool`

These are the same three tools as Exercise 2. The difference is how we declare them.

In Exercise 2 you hand-wrote a JSON schema (`name`, `description`, `parameters`) **and** a separate
Python function for each tool, then linked them through `tool_map`. With LangChain you write **only
the function**. The `@tool` decorator reads the function name, the type hints, and the docstring
and builds the schema for you. The docstring is not a comment here: it is the tool description the
model actually sees, so it has to be clear.

In [ ]:
@tool
def web_search(query: str) -> str:
    """Search the web for current information. Returns snippets with source URLs."""
    try:
        with DDGS() as ddgs:
            results = list(ddgs.text(query, max_results=5))
        if not results:
            return "No results found."
        return "\n".join(f"- {r['title']}: {r['body']} ({r['href']})" for r in results)
    except Exception as exc:
        return f"Search error: {exc}"


@tool
def calculator(expression: str) -> str:
    """Calculate a math expression. Supports +, -, *, /, **, sqrt, sin, cos, log, pi, e."""
    try:
        result = numexpr.evaluate(
            expression.strip(), global_dict={}, local_dict={"pi": math.pi, "e": math.e}
        )
        return f"Result: {float(result)}"
    except Exception as exc:
        return f"Error evaluating '{expression}': {exc}"


@tool
def read_file(filename: str) -> str:
    """Read a local text file from the project directory, e.g. 'sample.txt'."""
    safe_name = os.path.basename(filename)
    if not os.path.isfile(safe_name):
        return f"Error: File '{safe_name}' not found in project directory."
    with open(safe_name, "r", encoding="utf-8") as f:
        return f.read(10_000)


tools = [web_search, calculator, read_file]

# The decorator turned each function into a Tool object with an auto-generated schema.
print("tools:", [t.name for t in tools])
print("\ncalculator schema the model sees:")
print("  description:", calculator.description)
print("  args:", calculator.args)

## Step 2: Create the agent

Two objects:

- `ChatOllama` is the LangChain wrapper around the same `ollama.chat()` you used directly in
  Exercise 2. `temperature=0` makes tool calls deterministic.
- `create_agent(model, tools, system_prompt)` returns a ready-to-run agent. **This single call
  replaces the entire `run_agent` loop you wrote by hand.** It binds the tools to the model, runs
  the reason-act loop, executes tools, and feeds results back, until the model stops asking for
  tools.

In [ ]:
llm = ChatOllama(model=MODEL, temperature=0, num_ctx=32768)

SYSTEM_PROMPT = """You are a thorough research and engineering assistant with access to tools.

Rules:
- Always use calculator for ANY arithmetic, never compute in your head.
- Always use web_search for facts you are not certain about.
- Always use read_file when the user references a file.
- Never fabricate tool results. Cite your sources when using web_search."""

agent = create_agent(model=llm, tools=tools, system_prompt=SYSTEM_PROMPT)
print("agent ready")

## Step 3: A helper to watch the loop

`agent.invoke({...})` runs the whole loop and returns only the final state. To **see** the same
reason-act steps you printed by hand in Exercise 2, we stream the agent and print each message as
it arrives: a tool call the model decides to make, a tool result coming back, or the final answer.

In [ ]:
def run_agent(user_input: str) -> str:
    """Run the LangChain agent and print each step, like Exercise 2 did by hand."""
    print(f"\nQuestion: {user_input}")
    final = ""
    inputs = {"messages": [{"role": "user", "content": user_input}]}
    for chunk in agent.stream(inputs, stream_mode="updates"):
        for node_output in chunk.values():
            for msg in node_output.get("messages", []):
                if getattr(msg, "tool_calls", None):
                    for tc in msg.tool_calls:
                        print(f"  Tool: {tc['name']}({tc['args']})")
                elif getattr(msg, "type", None) == "tool":
                    print(f"  Result: {msg.content[:200]}")
                elif getattr(msg, "content", ""):
                    final = msg.content
                    print(f"  Final answer: {final[:300]}")
    return final

## Step 4: The same three tests as Exercise 2

Same questions, same tools, same model. Compare the output to your hand-built agent: the steps are
identical, you just did not have to write the loop.

In [ ]:
# Test 1: Calculator
answer = run_agent("Calculate (0.2 * 0.4**3) / 12")
print(f"\nResult: {answer}")

In [ ]:
# Test 2: Read a file
answer = run_agent("Read the file sample.txt and list the formulas it contains.")
print(f"\nResult: {answer}")

In [ ]:
# Test 3: Web search
answer = run_agent("Search the web for the Young's modulus of structural steel S235.")
print(f"\nResult: {answer}")

## Summary

You built the same agent twice. By hand (Exercise 2) you wrote the JSON schemas, the loop, the
`tool_calls` check, the dispatch, and the message bookkeeping. With LangChain all of that became:

```python
agent = create_agent(model=llm, tools=tools, system_prompt=SYSTEM_PROMPT)
```

**What the framework gave you:** schema generation from type hints and docstrings, the reason-act
loop, tool execution, and typed message handling, all tested and maintained.

**What you gave up:** transparency and control. When the agent does something surprising, the loop
is now inside the library. Knowing exactly what it does, because you wrote it once in Exercise 2,
is what lets you debug it.

That is the trade every agent framework makes: less code, less visibility. You now understand
both sides.